# CICIDS2017 : chargement et nettoyage
**Licence 3 TCO, ASJA Antsirabe** — suite du notebook NSL-KDD.

Objectif : obtenir un jeu **propre, léger et réutilisable** (`cicids2017_clean.parquet`) que tu pourras charger en 2 secondes dans tes futurs notebooks.

### Télécharger les données
Page officielle : https://www.unb.ca/cic/datasets/ids-2017.html → archive **MachineLearningCSV.zip** (8 fichiers CSV, un par demi-journée, ~230 Mo compressés).
Décompresse-la dans `data/raw/MachineLearningCVE/` (chemin par défaut de `DATA_DIR`).

> Ne prends **pas** la version « TrafficLabelling » pour le ML : elle contient IP, ports source et horodatages, qui font « tricher » le modèle (il apprend les adresses des attaquants au lieu du comportement).

## 0. Imports et configuration

In [ ]:
import os, glob, re, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = os.environ.get("CICIDS_DIR", "../data/raw/MachineLearningCVE")   # <- adapte ce chemin
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_FILE = f"{OUT_DIR}/cicids2017_clean.parquet"
SEED = 42
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(len(files), "fichiers trouvés")
for f in files: print(" -", os.path.basename(f))
assert files, "Aucun CSV trouvé : vérifie DATA_DIR"

## 1. Chargement, fichier par fichier
Pourquoi fichier par fichier ? Le jeu complet fait ~2,8 millions de lignes ; en `float64` il occupe ~2 Go de RAM.
On convertit chaque fichier en `float32`/`int32` **dès qu'il est lu** pour diviser l'empreinte mémoire par deux.

Deux pièges connus :
- les noms de colonnes ont des **espaces au début** (`" Flow Duration"`) → on les enlève ;
- les étiquettes contiennent un **caractère non ASCII** (`Web Attack � Brute Force`) → on lit en `latin-1` puis on corrige.

In [ ]:
def load_one(path):
    df = pd.read_csv(path, encoding="latin-1", low_memory=False)
    df.columns = df.columns.str.strip()
    # Colonnes numériques : tout sauf Label (certains fichiers ont des valeurs texte parasites)
    feat = [c for c in df.columns if c != "Label"]
    df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
    # Réduction mémoire
    for c in feat:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
        elif pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
    df["source_file"] = os.path.basename(path).split("-")[0]   # Monday, Tuesday...
    return df

frames = []
for f in files:
    d = load_one(f)
    print(f"{os.path.basename(f):60s} {d.shape[0]:>8,} lignes")
    frames.append(d)

# Les 8 fichiers n'ont normalement pas tous exactement les mêmes colonnes : on le vérifie
cols_ref = set(frames[0].columns)
for f, d in zip(files, frames):
    diff = set(d.columns) ^ cols_ref
    if diff: print("⚠️ colonnes différentes dans", os.path.basename(f), ":", diff)

df = pd.concat(frames, ignore_index=True)
del frames; gc.collect()
print("\nTotal :", df.shape)
print(f"Mémoire : {df.memory_usage(deep=True).sum()/1e6:.0f} Mo")

## 2. Inventaire initial
Avant de nettoyer, on **mesure** ce qui ne va pas. Chaque chiffre ici ira dans ton chapitre 3 (« Étude et préparation des données »).

In [ ]:
report = {"lignes_brutes": len(df), "colonnes_brutes": df.shape[1]}
print("Étiquettes brutes :")
print(df["Label"].value_counts().to_string())

## 3. Nettoyage, étape par étape
### 3.1 Étiquettes : réparer le caractère parasite

In [1]:
def clean_label(s):
    s = re.sub(r"[^\x20-\x7E]", "-", str(s))      # tout caractère non ASCII -> '-'
    s = re.sub(r"\s*-\s*", " - ", s) if s.startswith("Web Attack") else s
    return re.sub(r"\s+", " ", s).strip()

df["Label"] = df["Label"].map(clean_label)
print(sorted(df["Label"].unique()))

NameError: name 'df' is not defined

### 3.2 En-têtes répétés et lignes vides
Quand une conversion numérique échoue, la valeur devient `NaN` : on repère les lignes **entièrement** vides ou dont l'étiquette est vide.

In [2]:
feat_cols = [c for c in df.columns if c not in ("Label", "source_file")]
all_nan = df[feat_cols].isna().all(axis=1)
print("Lignes entièrement vides :", int(all_nan.sum()))
df = df[~all_nan].copy()
report["lignes_vides"] = int(all_nan.sum())

NameError: name 'df' is not defined

### 3.3 Valeurs infinies et manquantes
`Flow Bytes/s` et `Flow Packets/s` sont calculées par une division : quand la durée du flux est 0, on obtient `inf` ou `NaN`.
Elles ne représentent qu'une toute petite part des lignes, on les supprime.

In [3]:
n_inf = int(np.isinf(df[feat_cols].to_numpy(dtype="float32", na_value=np.nan)).any(axis=1).sum())
print("Lignes avec ±inf :", n_inf)
df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan)

na_per_col = df[feat_cols].isna().sum()
print("\nColonnes avec NaN :")
print(na_per_col[na_per_col > 0].to_string())

n_before = len(df)
df = df.dropna(subset=feat_cols).copy()
report["lignes_inf_nan"] = n_before - len(df)
print(f"\nLignes supprimées (inf/NaN) : {n_before - len(df):,}")

NameError: name 'np' is not defined

### 3.4 Valeurs impossibles (négatifs)
Certaines colonnes ne devraient jamais être négatives (durée, nombre de paquets, octets). D'autres, comme `Init_Win_bytes_forward`, valent légitimement `-1` (= « pas de fenêtre TCP observée »).
On supprime seulement les négatifs **réellement invalides**.

In [4]:
neg = (df[feat_cols] < 0).sum()
print("Colonnes contenant des valeurs négatives :")
print(neg[neg > 0].to_string())

MUST_BE_POSITIVE = [c for c in ["Flow Duration", "Total Fwd Packets", "Total Backward Packets",
                                "Flow Bytes/s", "Flow Packets/s"] if c in df.columns]
bad = (df[MUST_BE_POSITIVE] < 0).any(axis=1)
print(f"\nLignes à valeurs invalides supprimées : {int(bad.sum())}")
report["lignes_negatives_invalides"] = int(bad.sum())
df = df[~bad].copy()

NameError: name 'df' is not defined

### 3.5 Doublons de lignes
Le même flux peut apparaître plusieurs fois (surtout pour DoS et PortScan). Si un doublon tombe à la fois dans le train et le test, **le modèle est testé sur des exemples déjà vus** : fuite de données.
On compare sur toutes les caractéristiques **et** l'étiquette (pas sur `source_file`).

In [5]:
n_before = len(df)
df = df.drop_duplicates(subset=feat_cols + ["Label"]).reset_index(drop=True)
report["doublons"] = n_before - len(df)
print(f"Doublons supprimés : {n_before - len(df):,}  ({(n_before - len(df))/n_before:.1%})")

NameError: name 'df' is not defined

### 3.6 Colonnes inutiles : constantes et colonnes dupliquées
- une colonne constante n'apporte aucune information ;
- `Fwd Header Length.1` est une copie exacte de `Fwd Header Length` (bug connu du jeu).

In [6]:
const_cols = [c for c in feat_cols if df[c].nunique() <= 1]
print("Colonnes constantes :", const_cols)

# Colonnes identiques : on cherche sur un échantillon, puis on confirme sur tout le jeu
samp = df[feat_cols].sample(min(100_000, len(df)), random_state=SEED)
dup_cols = []
for i, a in enumerate(feat_cols):
    if a in dup_cols or a in const_cols: continue
    for b in feat_cols[i+1:]:
        if b in dup_cols or b in const_cols: continue
        if samp[a].equals(samp[b]) and df[a].equals(df[b]):
            dup_cols.append(b); print(f"  {b}  ==  {a}")
print("Colonnes dupliquées :", dup_cols)

drop = const_cols + dup_cols
df = df.drop(columns=drop)
report["colonnes_supprimees"] = len(drop)
feat_cols = [c for c in df.columns if c not in ("Label", "source_file")]
print(f"\nRestent {len(feat_cols)} caractéristiques")

NameError: name 'feat_cols' is not defined

## 4. Étiquettes pour le ML
Trois niveaux, selon la question posée :
- `is_attack` : binaire (normal / attaque) ;
- `family` : familles regroupées (utile pour la démo « type d'attaque probable ») ;
- `Label` : étiquette d'origine (15 classes).

In [7]:
def to_family(l):
    if l == "BENIGN": return "Benign"
    if l.startswith("DoS") or l == "DDoS" or l == "Heartbleed": return "DoS/DDoS"
    if l == "PortScan": return "PortScan"
    if l in ("FTP-Patator", "SSH-Patator"): return "BruteForce"
    if l.startswith("Web Attack"): return "WebAttack"
    if l == "Bot": return "Botnet"
    if l == "Infiltration": return "Infiltration"
    return "Autre"

df["is_attack"] = (df["Label"] != "BENIGN").astype("int8")
df["family"] = df["Label"].map(to_family)
print(df["family"].value_counts().to_string())
print("\nAutre (devrait être vide) :", int((df.family == "Autre").sum()))

NameError: name 'df' is not defined

## 5. Distribution des classes après nettoyage

In [8]:
fig, ax = plt.subplots(figsize=(8, 4.5))
vc = df["Label"].value_counts()
vc.plot.barh(ax=ax, logx=True, color="tab:blue")
ax.invert_yaxis(); ax.set_xlabel("Nombre de flux (échelle log)")
ax.set_title("CICIDS2017 après nettoyage")
plt.tight_layout(); plt.show()

print(f"Part d'attaques : {df.is_attack.mean():.1%}")
print("Classes très rares (< 100 exemples) :", vc[vc < 100].to_dict())

NameError: name 'plt' is not defined

## 6. Bilan du nettoyage
À recopier dans le mémoire.

In [9]:
report["lignes_finales"] = len(df)
report["colonnes_finales"] = df.shape[1]
summary = pd.Series(report, name="valeur").to_frame()
summary

NameError: name 'df' is not defined

## 7. Sauvegarde
Le format **Parquet** est ~5× plus petit que le CSV et se charge très vite. Sans `pyarrow`, on retombe sur un CSV compressé.

In [10]:
try:
    df.to_parquet(OUT_FILE, index=False)
    print("Sauvegardé :", OUT_FILE, f"({os.path.getsize(OUT_FILE)/1e6:.0f} Mo)")
except Exception as e:
    print("Parquet indisponible (pip install pyarrow) :", e)
    OUT_FILE = f"{OUT_DIR}/cicids2017_clean.csv.gz"
    df.to_csv(OUT_FILE, index=False)
    print("Sauvegardé :", OUT_FILE)

Parquet indisponible (pip install pyarrow) : name 'df' is not defined


NameError: name 'OUT_DIR' is not defined

## 8. Échantillon équilibré pour des essais rapides
Avec 2 millions de lignes, chaque essai est lent. On crée un échantillon de **max. 20 000 flux par classe** : toutes les classes rares sont gardées en entier, et les gros volumes (BENIGN, DoS Hulk, PortScan…) sont réduits.
Réserve-le au **développement** ; les résultats finaux du mémoire doivent venir du jeu complet.

In [11]:
CAP = 20_000
parts = [g.sample(min(len(g), CAP), random_state=SEED) for _, g in df.groupby("Label")]
sample = pd.concat(parts).reset_index(drop=True)
print(sample["Label"].value_counts().to_string())

if OUT_FILE.endswith(".parquet"):
    sample.to_parquet(f"{OUT_DIR}/cicids2017_sample.parquet", index=False)
else:
    sample.to_csv(f"{OUT_DIR}/cicids2017_sample.csv.gz", index=False)

NameError: name 'df' is not defined

## 9. Pour comparer avec NSL-KDD : variables équivalentes
Les deux jeux n'ont pas les mêmes colonnes. Voici les correspondances **approximatives** pour un test inter-datasets (entraîner sur l'un, tester sur l'autre).

| NSL-KDD | CICIDS2017 | Attention |
|---|---|---|
| `duration` (secondes) | `Flow Duration` (**microsecondes**) | diviser par 1 000 000 |
| `src_bytes` | `Total Length of Fwd Packets` | direction client → serveur |
| `dst_bytes` | `Total Length of Bwd Packets` | direction serveur → client |
| `protocol_type` (tcp/udp/icmp) | `Protocol` (6 / 17 / 1) | recoder en texte |
| `count`, `srv_count` | pas d'équivalent direct | fenêtre de 2 s dans NSL-KDD, à recalculer |
| `service` (http, ftp…) | `Destination Port` | à approximer (80 → http, 21 → ftp…) |

Au final, seules **4 à 6 variables** sont vraiment comparables : c'est une limite à expliquer honnêtement au jury, pas un défaut de ton travail.

### Prochaines étapes
1. Charger `cicids2017_clean.parquet` dans un nouveau notebook de modélisation
2. **Découpage** : stratifié aléatoire, ou **par jour** (entraîner lundi–jeudi, tester vendredi) pour une évaluation plus dure et plus réaliste
3. Reprendre les 4 modèles du notebook NSL-KDD
4. Gérer le déséquilibre (`class_weight`, sous-échantillonnage)